In [28]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json 
from statistics import mean

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [29]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset=generate_dataset()
print(dataset)

[{'task': 'Write a regular expression to validate AWS S3 bucket names. S3 bucket names must be 3-63 characters long, contain only lowercase letters, numbers, hyphens, and periods, start and end with a letter or number, and cannot contain consecutive hyphens or periods.'}, {'task': "Write a Python function that takes an AWS CloudWatch log stream name and returns a dictionary with the log group name and stream name separated. For example, 'my-app-logs/lambda/2024-01-15' should return {'log_group': 'my-app-logs/lambda', 'stream_name': '2024-01-15'}."}, {'task': "Generate a JSON object representing an AWS IAM policy that allows read-only access to all objects in an S3 bucket named 'company-data'. The policy should grant s3:GetObject and s3:ListBucket permissions for only that bucket."}]


In [ ]:
# save dataset to file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [34]:
# Helper functions for evaluating prompts

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    prompt=f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

def run_eval(dataset):
    """Loads the datase and calls run_test_case with each case"""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [35]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [36]:
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Write a regular expression to validate AWS S3 bucket names. S3 bucket names must be 3-63 characters long, contain only lowercase letters, numbers, hyphens, and periods, start and end with a letter or number, and cannot contain consecutive hyphens or periods."
    },
    "output": "# AWS S3 Bucket Name Validation Regex\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Single Regex Pattern (Recommended)\n\n```regex\n^(?!.*(-{2}|\\\\.{2}|\\\\.-|-.))(?!-)(?!.*-$)[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$\n```\n\n### Explanation:\n- `^` - Start of string\n- `(?!.*(-{2}|\\.{2}|\\.-|-\\.))` - Negative lookahead: no consecutive hyphens, periods, or mixed consecutive hyphens/periods\n- `(?!-)` - Negative lookahead: cannot start with hyphen\n- `(?!.*-$)` - Negative lookahead: cannot end with hyphen\n- `[a-z0-9]` - First character must be letter or number\n- `([a-z0-9.-]{1,61}[a-z0-9])?` - Optional: 1-61 middle characters (any valid),